# quantization_final — INT8-квантизация dense (дистиллят) и pruned (RecoveryKD) моделей на Kaggle CPU

**Что делает:** чекпоинт nnUNet → ONNX (opset 11) → OpenVINO IR FP32 → NNCF PTQ INT8 (MIXED) с калибровкой из b2nd-препроцессинга → верификация argmax INT8 vs FP32 + скорость.

**Два режима (Выбор в ячейке 2, `MODEL_KIND`):**
- `dense` — дистиллят `nnUNetTrainerDistill` (чекпоинт из Output ядра `nnunet-distill`);
- `pruned` — recovery-модель `nnUNetTrainerRecoveryKD` (чекпоинт из датасета `pruned-results`).

**Калибровка:** 25 случаев × 4 патча = 100 патчей (56,192,192) из b2nd-томов `alenasnigireva/nnunet`. Домен = aligned (b2nd уже препроцессированы nnUNet). b2nd хранит **(1, Z, Y, X)** — канал первой осью, Z со spacing 2.0 (самая короткая ось 150–180). Никаких транспонов (проверено: лишний transpose дал 0 патчей).

**Чекпоинтование/продолжение:** каждый этап сохраняет свой результат в `/kaggle/working` и при повторном запуске (Save & Run All после ошибки) пропускает уже сделанное: `stage_done()` проверяет файл-маркер. Калибровочные патчи тоже кэшируются в working — при перезапуске NNCF не пересчитывает калибровку, только квантизацию.

**Остановка при нештатном поведении:** страница ядра → ⋮ → **Stop** (не ждите ошибки). RAM печатается на каждом этапе; NNCF自适应ный K: >20 ГБ → 48 патчей, >10 → 24, иначе 12.

**Результат:** Output → `quantized_model.xml/.bin` (+ `model_meta.json` с именем тренера, эпохой, датой). Укладка: `models\headandneck\<имя>\fold_0\`.

In [ ]:
# ---------- Ячейка 1: окружение (пип-установки + RAM-монитор) ----------
import functools
import glob
import json
import os
import subprocess
import sys
import time

print = functools.partial(print, flush=True)

def avail_gb():
    with open('/proc/meminfo') as f:
        for line in f:
            if line.startswith('MemAvailable'):
                return int(line.split()[1]) / 2**20
    return -1

def ram():
    return f'{avail_gb():.1f} ГБ свободно'

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
sys.modules.setdefault('tensorflow', None)   # NNCF иначе пытается импортировать TF
sys.modules.setdefault('tensorboard', None)

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

# ---------- стек квантизации (Этап 1: эксперимент нового стека) ----------
STACK = 'prod'  # 'prod' = nncf 2.19 + ov 2025.2 (прод-пины, валидировано)
# 'v3x' = nncf 3.4 + ov 2026.4 — эксперимент 2026-10-23: гейт идентичен (0.9995),
# выигрыша нет — см. BENCHMARKS.md, раздел «Стек квантизации»
WANT_NNCF = '3.4.0' if STACK == 'v3x' else '2.19.0'
WANT_OV = '2026.4.1' if STACK == 'v3x' else '2025.2.0'
print(f'[stack] {STACK}: цель nncf=={WANT_NNCF}, openvino=={WANT_OV}')

try:
    import nncf
except ImportError:
    print(f'[stack] nncf отсутствует — ставлю {WANT_NNCF} ...')
    _pip(f'nncf=={WANT_NNCF}')
    import nncf
if nncf.__version__ != WANT_NNCF:
    print(f'[stack] nncf {nncf.__version__} -> {WANT_NNCF} ...')
    _pip(f'nncf=={WANT_NNCF}')
    import importlib as _iln
    _iln.reload(nncf)
print('[stack] nncf', nncf.__version__)

try:
    import openvino as ov
except ImportError:
    print(f'[stack] openvino отсутствует — ставлю {WANT_OV} ...')
    _pip(f'openvino=={WANT_OV}')
    import openvino as ov
if ov.__version__ != WANT_OV:
    print(f'[stack] openvino {ov.__version__} -> {WANT_OV} ...')
    _pip(f'openvino=={WANT_OV}')
    import importlib as _ilo
    _ilo.reload(ov)
print('[stack] openvino', ov.__version__)

try:
    import nnunetv2
    if not hasattr(nnunetv2, '__version__'):
        nnunetv2.__version__ = '2.6.2'
except ImportError:
    _pip('nnunetv2==2.6.2')

# onnxscript нужен dynamo-экспортёру torch.onnx в torch>=2.9 (на Kaggle-образе
# с torch 2.11 его нет — ModuleNotFoundError при torch.onnx.export)
try:
    import onnxscript
except ImportError:
    print('ставлю onnxscript (нужен ONNX-экспортёру torch 2.11) ...')
    _pip('onnxscript')

import numpy as np
import torch
import openvino as ov
import nncf
print('openvino', ov.__version__, '| torch', torch.__version__, '| nncf', nncf.__version__)
print('RAM:', ram())

WORK = '/kaggle/working'

def stage_done(name):
    """Файл-маркер завершённого этапа: при повторном запуске этап пропускается."""
    return os.path.isfile(os.path.join(WORK, f'.stage_{name}_done'))

def mark_stage(name):
    open(os.path.join(WORK, f'.stage_{name}_done'), 'w').write('ok')

In [ ]:
# ---------- Ячейка 2: ВЫБОР МОДЕЛИ ----------
# 'dense'  — дистиллят nnUNetTrainerDistill (чекпоинт из Output ядра nnunet-distill)
# 'pruned' — recovery nnUNetTrainerRecoveryKD (чекпоинт из датасета pruned-results)
MODEL_KIND = 'pruned'          # <-- ПЕРЕКЛЮЧАТЕЛЬ: 'dense' или 'pruned'

STAMP = time.strftime('%Y%m%d_%H%M%S')
print('MODEL_KIND =', MODEL_KIND, '| штамп:', STAMP)
assert MODEL_KIND in ('dense', 'pruned')

In [ ]:
# ---------- Ячейка 3: найти чекпоинт и plans.json выбранной модели ----------
def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

def find_any(*patterns):
    for p in patterns:
        h = find_one(p)
        if h:
            return h
    return None

if MODEL_KIND == 'dense':
    CKPT = find_any('/kaggle/input/**/nnUNetTrainerDistill__*/fold_0/checkpoint_final.pth',
                    '/kaggle/input/**/nnUNetTrainerDistill__*/fold_0/checkpoint_best.pth')
    PLANS = find_any('/kaggle/input/**/nnUNetTrainerDistill__*/plans.json')
    TRAINER_TAG = 'nnUNetTrainerDistill'
else:
    CKPT = find_any('/kaggle/input/**/datasets/**/nnUNetTrainerRecoveryKD__*/fold_0/checkpoint_final.pth',
                    '/kaggle/input/**/datasets/**/nnUNetTrainerRecoveryKD__*/fold_0/checkpoint_best.pth')
    PLANS = find_any('/kaggle/input/**/datasets/**/nnUNetTrainerRecoveryKD__*/plans.json')
    TRAINER_TAG = 'nnUNetTrainerRecoveryKD'

assert CKPT, (f'чекпоинт {TRAINER_TAG} не найден. Дерево input — в следующей ячейке. '
              'dense: должно быть приаттачено ядро nnunet-distill; '
              'pruned: датасет pruned-results (заливается через сайт).')
assert PLANS, f'plans.json {TRAINER_TAG} не найден рядом с чекпоинтом'

import torch
ck = torch.load(CKPT, map_location='cpu', weights_only=False)
epoch = ck.get('current_epoch', '?')
trainer_name = ck.get('trainer_name', '?')
print('чекпоинт:', CKPT)
print('тренер:', trainer_name, '| эпоха:', epoch)
if MODEL_KIND == 'dense':
    assert 'Distill' in str(trainer_name), f'это не студент: {trainer_name}'
else:
    assert 'Recovery' in str(trainer_name), f'это не recovery-модель: {trainer_name}'
print('RAM:', ram())

In [ ]:
# ---------- Ячейка 4: диагностика монтирований (видна всегда) ----------
print('дерево /kaggle/input (4 уровня, без b2nd/pkl/архивов):')
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root[len('/kaggle/input'):].count(os.sep)
    if depth >= 4:
        dirs[:] = []
    show = [f for f in files if not f.endswith(('.b2nd', '.pkl'))][:6]
    if show or depth <= 1:
        print(' ', root, '->', show)

In [ ]:
# ---------- Ячейка 5: калибровочные патчи из b2nd (кэш в working) ----------
# Этап чекпоинтуется: calibration_patches.npz в /kaggle/working — при повторном
# запуске (после ошибки в NNCF) калибровка НЕ пересчитывается.
import blosc2
import random

CACHE = os.path.join(WORK, 'calibration_patches.npz')
if stage_done('calib') and os.path.isfile(CACHE):
    X_calib = np.load(CACHE)['patches']
    print(f'кэш калибровки: {X_calib.shape} из {CACHE}')
else:
    DATA_DIR = find_one('/kaggle/input/**/nnUNet_preprocessed/Dataset001_headandneck/*_3d_fullres')
    assert DATA_DIR, 'не найден препроцессинг b2nd (нужен kernel_source alenasnigireva/nnunet)'
    cases = sorted(os.path.splitext(os.path.basename(f))[0]
                   for f in glob.glob(os.path.join(DATA_DIR, '*.b2nd'))
                   if not f.endswith('_seg.b2nd'))
    print('папка b2nd:', DATA_DIR, '| случаев:', len(cases))

    PATCH = (56, 192, 192)       # (Z, Y, X) из plans 3d_fullres
    N_CASES, PER_CASE = 25, 4    # 100 патчей × 8.3 МБ ≈ 0.9 ГБ
    rng = random.Random(20261012)
    sel = rng.sample(cases, min(N_CASES, len(cases)))
    patches = []
    t0 = time.time()
    for i, case in enumerate(sel):
        arr = np.asarray(blosc2.open(os.path.join(DATA_DIR, case + '.b2nd')))
        if arr.ndim == 4 and arr.shape[0] == 1:
            arr = arr[0]                      # (1, Z, Y, X) -> (Z, Y, X)
        assert arr.ndim == 3, f'неожиданная форма b2nd: {arr.shape}'
        assert arr.shape[0] < arr.shape[1], (
            f'ось 0 не похожа на Z (spacing 2.0, 150-180 срезов): {arr.shape}')
        zmax = max(1, arr.shape[0] - PATCH[0])
        for _ in range(PER_CASE):
            z = rng.randint(0, zmax)
            y = rng.randint(0, max(1, arr.shape[1] - PATCH[1]))
            x = rng.randint(0, max(1, arr.shape[2] - PATCH[2]))
            p = arr[z:z + PATCH[0], y:y + PATCH[1], x:x + PATCH[2]]
            if p.shape == PATCH:
                patches.append(p.astype(np.float32))
        if (i + 1) % 10 == 0 or i == len(sel) - 1:
            print(f'[{i+1}/{len(sel)}] {case}: {arr.shape} -> {len(patches)} патчей, '
                  f'{time.time()-t0:.0f} с, {ram()}')
    assert patches, 'патчи не вырезались — проверить оси b2nd'
    X_calib = np.stack(patches)[:, np.newaxis]
    assert abs(X_calib.mean()) < 20, 'данные не похожи на CTNormalization-домен'
    np.savez_compressed(CACHE, patches=X_calib)
    mark_stage('calib')
    print('кэш сохранён:', CACHE)

print('итог:', X_calib.shape, f'домен mean={X_calib.mean():.3f} std={X_calib.std():.3f}')
print('RAM:', ram())

In [ ]:
# ---------- Ячейка 6: чекпоинт -> сеть -> ONNX -> OpenVINO IR (FP32) ----------
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunetv2.utilities.label_handling.label_handling import determine_num_input_channels

plans = json.load(open(PLANS))
cfg = plans['configurations']['3d_fullres']
features = cfg['architecture']['arch_kwargs'].get('features_per_stage')
patch_size = tuple(cfg['patch_size'])
print('patch_size:', patch_size, '| features_per_stage:', features)
if MODEL_KIND == 'pruned':
    assert features[0] < 64, 'у pruned-модели первый блок должен быть уже dense'

# dataset.json — детерминированная реконструкция из меток проекта (31 орган)
organ_labels = ['BrachialPlexus_R', 'BrachialPlexus_L', 'SpinalCanal', 'SpinalCord',
                'Trachea', 'Esophagus', 'Thyroid', 'A_Carotid_R', 'A_Carotid_L',
                'Larynx', 'PharynxConst_I', 'PharynxConst_M', 'PharynxConst_S',
                'OpticNerve_R', 'OpticNerve_L', 'Brain', 'Submandibular_R',
                'Submandibular_L', 'Parotid_R', 'Parotid_L', 'Lips', 'Tongue',
                'Mandible', 'Eye_R', 'Lens_R', 'Eye_L', 'Lens_L', 'TMJoint_R',
                'TMJoint_L', 'Cochlea_R', 'Cochlea_L']
labels = {'background': 0}
labels.update({n: i + 1 for i, n in enumerate(organ_labels)})
dataset_json = {'channel_names': {'0': 'ct'}, 'labels': labels,
                'numTraining': 100, 'file_ending': '.nii.gz'}

pm = PlansManager(plans)
cm = pm.get_configuration('3d_fullres')
lm = pm.get_label_manager(dataset_json)
num_in = determine_num_input_channels(pm, cm, dataset_json)
net = nnUNetTrainer.build_network_architecture(
    cm.network_arch_class_name, cm.network_arch_init_kwargs,
    cm.network_arch_init_kwargs_req_import, num_in,
    lm.num_segmentation_heads, True)
state = ck.get('network_weights', ck.get('state_dict', ck))
missing, unexpected = net.load_state_dict(state, strict=False)
print(f'load_state_dict: missing={len(missing)}, unexpected={len(unexpected)}')
assert not missing, f'не хватило весов: {missing[:5]}'
net.eval()

# ONNX opset 18 (dynamo-экспортёр torch 2.11; opset 11 не достижим —
# version converter падает на Shape из opset 15). OV 2026.4 понимает opset 18.
t0 = time.time()
dummy = torch.randn(1, 1, *patch_size)
torch.onnx.export(net, dummy, 'student.onnx', export_params=True,
                  opset_version=18, do_constant_folding=True,
                  input_names=['input'], output_names=['output'])
print(f'ONNX: {time.time()-t0:.0f} с')

# external data (веса в .data) для Netron неудобны — переэкспорт single-file
if os.path.isfile('student.onnx.data'):
    print('!! веса в external data — переэкспортирую single-file')
    os.remove('student.onnx')
    os.remove('student.onnx.data')
    reexported = False
    try:  # dynamo-экспортёр (torch>=2.9 на Kaggle-образе)
        torch.onnx.export(net, dummy, 'student.onnx', export_params=True,
                          opset_version=18, do_constant_folding=True,
                          input_names=['input'], output_names=['output'],
                          external_data=False, dynamo=True)
        reexported = True
    except TypeError:
        pass
    if not reexported:  # legacy-экспортёр
        torch.onnx.export(net, dummy, 'student.onnx', export_params=True,
                          opset_version=18, do_constant_folding=True,
                          input_names=['input'], output_names=['output'],
                          save_as_external_data=False)
    assert not os.path.isfile('student.onnx.data'), 'веса всё ещё external'

ov_model = ov.convert_model('student.onnx')
ov.serialize(ov_model, 'student_fp32.xml')
print('IR FP32 готов, RAM:', ram())

In [ ]:
# ---------- Ячейка 7: NNCF INT8 (в subprocess — nbclient-таймауты не действуют) ----------
# Kaggle-nbclient рвёт долгую ячейку по iopub-таймауту (v5-v7: CellTimeoutError
# на 100-патчевой NNCF). Обход: NNCF выполняется отдельным python-процессом
# (runner_quant.py), stdout стримится читателем — таймауты не страшны, прогресс живой.
runner_code = chr(10).join([
    'import functools, json, sys, time',
    'print = functools.partial(print, flush=True)',
    'import numpy as np',
    'import os',
    "os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')",
    "sys.modules.setdefault('tensorflow', None)",
    "sys.modules.setdefault('tensorboard', None)",
    'import nncf',
    'import openvino as ov',
    "print(f'[runner] стек: nncf {nncf.__version__} | openvino {ov.__version__}')",
    'K = int(sys.argv[1])',
    'out_xml = sys.argv[2]',
    "calib = np.load('calibration_patches.npz')['patches']",
    'K = min(K, len(calib))',
    "print(f'runner: NNCF на {K} патчах из {len(calib)}')",
    'core = ov.Core()',
    "model = core.read_model('student_fp32.xml')",
    'def gen():',
    '    for i in range(K):',
    '        print(f\"[NNCF] патч {i+1}/{K}\", flush=True)',
    '        yield calib[i][np.newaxis]',
    't0 = time.time()',
    'q = nncf.quantize(model, nncf.Dataset(gen()),',
    '                  preset=nncf.QuantizationPreset.MIXED, subset_size=K)',
    'print(f\"runner: NNCF {time.time()-t0:.0f} с\")',
    'ov.serialize(q, out_xml)',
    "print('runner: сериализовано', out_xml)",
])
with open('runner_quant.py', 'w') as f:
    f.write(runner_code)

import subprocess
free = avail_gb()
CALIB_OVERRIDE = 48    # прямая сравнимость с v4 (nncf 2.19, K=48)
K = CALIB_OVERRIDE if CALIB_OVERRIDE else (48 if free > 20 else (24 if free > 10 else 12))
K = min(K, len(X_calib))
print(f'свободно {free:.1f} ГБ -> калибровка на {K} патчах')
out_xml = f'quantized_model_{STACK}_{K}p.xml' if CALIB_OVERRIDE else f'quantized_model_{STACK}.xml'
print(f'запуск runner: K={K} -> {out_xml}')
proc = subprocess.Popen([sys.executable, 'runner_quant.py', str(K), out_xml],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for ln in proc.stdout:
    print(ln.rstrip())
proc.wait()
assert proc.returncode == 0 and os.path.isfile(out_xml), (
    f'runner упал (код {proc.returncode}) — см. вывод выше')
print('RAM после runner:', ram())

In [ ]:
# ---------- Ячейка 8: верификация argmax INT8 vs FP32 (гейт 0.9) ----------
import openvino as _ov
core = _ov.Core()
fp32 = core.compile_model(core.read_model('student_fp32.xml'), 'CPU')
int8 = core.compile_model(core.read_model(out_xml), 'CPU')
agree_all = []
for i in range(min(8, len(X_calib))):
    p = X_calib[i][np.newaxis]
    a = fp32(p)[fp32.output(0)][0].argmax(0)
    b = int8(p)[int8.output(0)][0].argmax(0)
    agree_all.append(float((a == b).mean()))
    print(f'патч {i}: совпадение argmax {agree_all[-1]:.3f}')
    del a, b, p
mean_agree = float(np.mean(agree_all))
print(f'СРЕДНЕЕ совпадение argmax: {mean_agree:.4f}')
if mean_agree < 0.9:
    print('!! ГЕЙТ НЕ ПРОЙДЕН — квантизация сомнительна; проверьте домен калибровки')
else:
    print('ГЕЙТ ПРОЙДЕН')

In [ ]:
# ---------- Ячейка 9: скорость + метаданные + итог ----------
inp = np.zeros((1, 1) + patch_size, np.float32)
times = {}
for c_, tag in ((fp32, 'FP32'), (int8, f'INT8({K}п)')):
    for _ in range(2):
        c_(inp)
    t0 = time.perf_counter()
    for _ in range(5):
        c_(inp)
    times[tag] = (time.perf_counter() - t0) / 5
    print(f'{tag}: {times[tag]:.2f} с/патч (Kaggle CPU 4 vCPU)')
t32 = next(v for k, v in times.items() if k.startswith("FP32"))
t8 = next(v for k, v in times.items() if k.startswith("INT8"))
print(f'ускорение: {t32/t8:.2f}x')

meta = {'model_kind': MODEL_KIND, 'trainer_name': str(trainer_name),
        'epoch': str(epoch), 'checkpoint': CKPT, 'plans': PLANS,
        'features_per_stage': features, 'patch_size': list(patch_size),
        'calib_patches': int(len(X_calib)), 'argmax_agree': mean_agree,
        'sec_per_patch': times, 'timestamp': STAMP}
with open('model_meta.json', 'w') as f:
    json.dump(meta, f, indent=1)

print()
print(f'===== ГОТОВО: {MODEL_KIND} ({TRAINER_TAG}, эпоха {epoch}) =====')
print('Скачайте из Output: quantized_model.xml, quantized_model.bin, model_meta.json')
print('Укладка: models\\headandneck\\<имя>\\fold_0\\ (рядом quantized_model.xml/bin)')
print('RAM финал:', ram())